In [1]:
%reload_ext autoreload
%autoreload 2

import pandas as pd

pd.options.display.max_columns, pd.options.display.max_rows = 500, 200
pd.options.display.float_format = '{:.1f}'.format
import joblib
import warnings
warnings.filterwarnings('ignore')

from soccer.preprocessing import DataPreprocessor
from soccer.glicko_soccer import GlickoSoccer

Preprocessing

In [2]:
raw_matches = pd.read_csv('data/matches.csv')
raw_matches.head()

,date,home_team,away_team,home_score,away_score,notes,country,season,tournament_type
0,30.05.2009,Antalyaspor,Ankaragucu,1,0,NaN,Turkey,2008-2009,first
1,30.05.2009,Basaksehir,Bursaspor,0,1,NaN,Turkey,2008-2009,first
2,30.05.2009,Denizlispor,Besiktas,1,2,NaN,Turkey,2008-2009,first
3,30.05.2009,Galatasaray,Sivasspor,2,1,NaN,Turkey,2008-2009,first
4,30.05.2009,Genclerbirligi,Kayserispor,0,4,NaN,Turkey,2008-2009,first


In [3]:
raw_matches.shape

(591800, 9)

Fit parameters

In [4]:
matches = DataPreprocessor(is_actual_draw_predictions=False, max_season=2023, is_boosting_train=False).preprocessing()

In [91]:
# it works a lot of time
# league_params = GlickoSoccer().fit_params(matches, number_iterations=100, is_params_initialization=True)
# league_params

Ratings

In [5]:
league_params = joblib.load('data/league_params.pkl')
club_ratings = GlickoSoccer().rate_teams(matches, league_params)
club_ratings = GlickoSoccer().ratings_to_df(club_ratings, matches)
europe_leagues_ratings, south_america_leagues_ratings = GlickoSoccer().league_ratings(club_ratings, number_top_teams=5)

club_ratings.head(50)

,#,team,rating,league,is_europe
0,1,Manchester City,2253.6,England. First,True
1,2,Real Madrid,2207.8,Spain. First,True
2,3,Inter,2207.3,Italy. First,True
3,4,Bayern Munich,2182.3,Germany. First,True
4,5,Liverpool,2177.1,England. First,True
5,6,Bayer Leverkusen,2166.5,Germany. First,True
6,7,Arsenal,2144.1,England. First,True
7,8,PSV,2110.1,Netherlands. First,True
8,9,FC Porto,2093.1,Portugal. First,True
9,10,Sporting CP,2090.5,Portugal. First,True


In [6]:
europe_leagues_ratings

,#,league,rating
0,1,England. First,2131.6
1,2,Germany. First,2094.0
2,3,Italy. First,2080.3
3,4,Spain. First,2069.3
4,5,Portugal. First,2006.5
5,6,Netherlands. First,1970.1
6,7,France. First,1962.7
7,8,Belgium. First,1951.1
8,9,Turkey. First,1939.5
9,10,Russia. First,1918.0


In [7]:
south_america_leagues_ratings

,#,league,rating
0,1,Brazil. First,1846.3
1,2,Argentina. First,1803.9
2,3,Paraguay. First,1740.5
3,4,Ecuador. First,1728.8
4,5,Colombia. First,1682.2
5,6,Uruguay. First,1670.1
6,7,Brazil. Second,1656.1
7,8,Chile. First,1629.9
8,9,Argentina. Second,1618.0
9,10,Peru. First,1599.7


In [8]:
from soccer.compare_models import compare_models

predictions = compare_models(start_season=2021)
predictions.head()

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.35, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.35
[LightGBM] [Warning] lambda_l1 is set=2.05114, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2.05114
[LightGBM] [Warning] lambda_l2 is set=7.23524, reg_lambda=0.0 will be ignored. Current value: lambda_l2=7.23524
[LightGBM] [Warning] bagging_fraction is set=0.772, subsample=1.0 will be ignored. Current value: bagging_fraction=0.772
[LightGBM] [Warning] bagging_freq is set=2, subsample_freq=0 will be ignored. Current value: bagging_freq=2


,model,loss
0,Catboost,0.6
1,LightGBM,0.6
2,Modified Glicko-2,0.6
3,Original Glicko-2,0.6
